In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
CSV_PATH = r"C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142.csv"

In [3]:
ENERGY_COL = "joules"

In [4]:
GROUP_COLS = ["language", "algorithm", "size"]

In [6]:
df = pd.read_csv(CSV_PATH)
 
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Columns: {list(df.columns)}\n")
 

Loaded 5400 rows from C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142.csv
Columns: ['language', 'algorithm', 'size', 'run', 'joules', 'checksum']



In [ ]:
stats = df.groupby(GROUP_COLS)[ENERGY_COL].agg(
        count="count",
        mean="mean",
        std="std"
    ).reset_index()

stats["cv_pct"] = (stats["std"] / stats["mean"]) * 100

In [9]:
 # --- Overall summary ---
print("=" * 60)
print("OVERALL CV SUMMARY")
print("=" * 60)
print(f"  Min  CV: {stats['cv_pct'].min():.2f}%  "
      f"→ {stats.loc[stats['cv_pct'].idxmin(), GROUP_COLS].to_dict()}")
print(f"  Max  CV: {stats['cv_pct'].max():.2f}%  "
      f"→ {stats.loc[stats['cv_pct'].idxmax(), GROUP_COLS].to_dict()}")
print(f"  Mean CV: {stats['cv_pct'].mean():.2f}%")
print(f"  Med  CV: {stats['cv_pct'].median():.2f}%")

OVERALL CV SUMMARY
  Min  CV: 4.40%  → {'language': 'python', 'algorithm': 'matrix_multiplication', 'size': 'mid'}
  Max  CV: 122.66%  → {'language': 'javascript', 'algorithm': 'merge_sort', 'size': 'small'}
  Mean CV: 38.38%
  Med  CV: 43.03%


In [10]:
# --- Top 10 highest variance cells ---
print("\n" + "=" * 60)
print("TOP 10 HIGHEST CV CELLS (most noisy)")
print("=" * 60)
print(stats.sort_values("cv_pct", ascending=False)
               .head(10)
               .to_string(index=False))


TOP 10 HIGHEST CV CELLS (most noisy)
  language     algorithm  size  count     mean      std     cv_pct
javascript    merge_sort small     50 0.012110 0.014854 122.663841
         c     summation   mid     50 0.004109 0.002679  65.208285
         c     summation small     50 0.004357 0.002733  62.722040
        go           bfs   mid     50 0.046465 0.029116  62.662226
      java binary_search   mid     50 0.010686 0.006444  60.305088
      rust    hash_table   mid     50 0.006724 0.004050  60.233184
    python binary_search small     50 0.004910 0.002904  59.137206
      rust binary_search large     50 0.003090 0.001738  56.266573
         c           bfs small     50 0.003336 0.001869  56.029696
        go binary_search   mid     50 0.004759 0.002664  55.982426


In [11]:
# --- Top 10 lowest variance cells ---
print("\n" + "=" * 60)
print("TOP 10 LOWEST CV CELLS (most stable)")
print("=" * 60)
print(stats.sort_values("cv_pct", ascending=True)
               .head(10)
               .to_string(index=False))


TOP 10 LOWEST CV CELLS (most stable)
  language             algorithm  size  count       mean       std   cv_pct
    python matrix_multiplication   mid     50  22.069947  0.970790 4.398695
    python                   bfs large     50  33.387005  1.797884 5.384981
javascript matrix_multiplication large     50  10.609192  0.599989 5.655367
      java            merge_sort large     50   0.816086  0.048040 5.886695
        go matrix_multiplication large     50   7.372315  0.482491 6.544632
    python matrix_multiplication large     50 559.995289 38.582423 6.889776
    python            merge_sort large     50  83.677411  5.801188 6.932801
javascript            merge_sort large     50   3.147572  0.218618 6.945597
      rust            merge_sort large     50   0.841862  0.071108 8.446467
    python            hash_table large     50   1.507898  0.135861 9.009979


In [13]:
# by lanbguage

print("\n" + "=" * 60)
print("MEAN CV BY LANGUAGE")
print("=" * 60)
lang_cv = stats.groupby("language")["cv_pct"].agg(
    mean_cv="mean",
    max_cv="max",
    min_cv="min"
).sort_values("mean_cv")
print(lang_cv.to_string())


MEAN CV BY LANGUAGE
              mean_cv      max_cv     min_cv
language                                    
python      28.849391   59.137206   4.398695
javascript  36.397647  122.663841   5.655367
java        38.476402   60.305088   5.886695
go          40.268383   62.662226   6.544632
c           42.305694   65.208285  10.158538
rust        43.966056   60.233184   8.446467


In [14]:
 # --- CV by algorithm ---
print("\n" + "=" * 60)
print("MEAN CV BY ALGORITHM")
print("=" * 60)
algo_cv = stats.groupby("algorithm")["cv_pct"].agg(
    mean_cv="mean",
    max_cv="max",
    min_cv="min"
).sort_values("mean_cv")
print(algo_cv.to_string())


MEAN CV BY ALGORITHM
                         mean_cv      max_cv     min_cv
algorithm                                              
matrix_multiplication  27.653939   52.162205   4.398695
bfs                    34.805932   62.662226   5.384981
merge_sort             35.942235  122.663841   5.886695
summation              40.712130   65.208285  10.205118
hash_table             42.701986   60.233184   9.009979
binary_search          48.447351   60.305088  36.333480


In [15]:
# --- Full table sorted by CV descending ---
print("\n" + "=" * 60)
print("FULL CV TABLE (sorted highest to lowest)")
print("=" * 60)
print(stats.sort_values("cv_pct", ascending=False)
               .to_string(index=False))


FULL CV TABLE (sorted highest to lowest)
  language             algorithm  size  count       mean       std     cv_pct
javascript            merge_sort small     50   0.012110  0.014854 122.663841
         c             summation   mid     50   0.004109  0.002679  65.208285
         c             summation small     50   0.004357  0.002733  62.722040
        go                   bfs   mid     50   0.046465  0.029116  62.662226
      java         binary_search   mid     50   0.010686  0.006444  60.305088
      rust            hash_table   mid     50   0.006724  0.004050  60.233184
    python         binary_search small     50   0.004910  0.002904  59.137206
      rust         binary_search large     50   0.003090  0.001738  56.266573
         c                   bfs small     50   0.003336  0.001869  56.029696
        go         binary_search   mid     50   0.004759  0.002664  55.982426
      rust            hash_table small     50   0.003338  0.001868  55.957520
        go            

In [16]:
# --- Save to CSV for reference ---
out_path = CSV_PATH.replace(".csv", "_cv_results.csv")
stats.sort_values("cv_pct", ascending=False).to_csv(out_path, index=False)
print(f"\nFull results saved to: {out_path}")


Full results saved to: C:\Users\Stemadmin\Desktop\Anuj Khadka\carbon_footprint\results\pilot_20260424_001142_cv_results.csv


In [18]:
max_row = stats.loc[stats["cv_pct"].idxmax()]
second_row = stats.sort_values("cv_pct", ascending=False).iloc[1]
c_rust = stats[stats["language"].isin(["c", "C", "rust", "Rust"])]
c_rust_max = c_rust["cv_pct"].max()

In [20]:
print("\n" + "=" * 60)
print("PAPER-READY SUMMARY (paste into section 5.4)")
print("=" * 60)
print(f"Per-cell CV ranged from {stats['cv_pct'].min():.1f}% to \n"
      f"{stats['cv_pct'].max():.1f}%, with the highest variance cells \n"
      f"being {max_row['language'].capitalize()} \n"
      f"{max_row['algorithm']} ({max_row['cv_pct']:.1f}% CV) and \n"
      f"{second_row['language'].capitalize()} \n"
      f"{second_row['algorithm']} ({second_row['cv_pct']:.1f}% CV). \n"
      f"Compiled language cells (C, Rust) showed CV values consistently \n"
      f"below {np.ceil(c_rust_max * 10) / 10:.1f}%.\n")


PAPER-READY SUMMARY (paste into section 5.4)
Per-cell CV ranged from 4.4% to 
122.7%, with the highest variance cells 
being Javascript 
merge_sort (122.7% CV) and 
C 
summation (65.2% CV). 
Compiled language cells (C, Rust) showed CV values consistently 
below 65.3%.

